# Lecture 03：Git 与 GitHub 总结及实操

原始资料：[Lecture 03 Ver2.docx](source/Lecture%2003%20Ver2.docx)。

**学习目标：** 区分 Git、GitHub、Desktop；理解工作区、暂存区、提交；完成分支、合并和冲突处理；区分撤销方式；观察 push、fetch、pull。

**使用方法：** Python 3.10+，另需已安装 Git 并加入 PATH。按顺序运行；重新实验请 Restart Kernel and Run All。Python 代码只使用标准库。Git 实验全部位于新建的临时目录，远程用本地 bare 仓库模拟，无需账号或联网。提交哈希和临时路径每次可能不同。


## 1. 工具与核心概念

| 名称 | 作用 |
|---|---|
| Git | 分布式版本控制工具；本地提交、分支、合并不需要联网 |
| GitHub | 托管 Git 仓库，支持团队协作、Pull Request、评审 |
| GitHub Desktop | 执行常见 Git 操作的图形界面客户端 |

~~~text
工作区 -- git add --> 暂存区 -- git commit --> 本地仓库
                                              |
                                          git push
                                              v
                                          远程仓库
远程 -- git fetch --> 远程跟踪引用 -- merge/rebase --> 当前分支
~~~

- 工作区：当前编辑的项目文件。
- 暂存区（index）：下一次提交准备记录的内容。
- commit：文件快照与作者、说明、父提交等信息；先保存在本地。
- branch：指向开发线上最新提交的可移动引用。
- HEAD：通常指向当前分支。origin：常用的远程名称，并非固定关键字。
- .git：保存版本历史和仓库元数据，不应随意删除。
- 保存文件、commit、push 是三个不同动作。


## 2. GitHub Desktop 操作速查

以下菜单名称按讲义整理，具体文字可能随界面版本变化。

| 任务 | 讲义中的操作 | 理解重点 |
|---|---|---|
| 安装与登录 | 安装 Desktop，在 Accounts 登录 GitHub | 讲义介绍 Windows/macOS 流程 |
| 基本设置 | Git 设置作者姓名/邮箱；Integrations 选择编辑器 | 提交署名和账号认证不同 |
| 创建仓库 | File → New Repository，选择名称、路径、README、ignore、license | 创建的是本地仓库 |
| 克隆仓库 | File → Clone Repository，选择仓库或 URL | 复制历史与工作区 |
| 提交修改 | Changes 检查差异、勾选内容、填写说明、Commit | 绿色表示增加，红色表示删除；只提交所选内容 |
| 查看历史 | History | 阅读提交说明、差异和提交哈希 |
| 上传 | Publish repository / Push origin | 发布仓库或上传后续本地提交 |
| 更新 | Fetch origin，再按需要 Pull origin | 获取与整合更新是不同步骤 |
| 分支 | Current Branch → New Branch | 独立开发一项功能 |
| 合并 | 先切换到目标分支，再 Merge into current branch | 当前分支接收修改 |
| Pull Request | 推送功能分支后 Create Pull Request | 在 GitHub 发起评审，不等于已合并 |

一次提交尽量只表达一个明确的修改目的。History 常显示缩短的哈希，它的长度不必固定为七位。


## 3. 建立临时实验环境

下面封装真实 Git 命令。参数通过列表传入，适用于含空格的 Windows 路径。命令只允许在本次临时目录内执行；作者配置写入练习仓库。实验子进程不读取全局 Git 配置，使签名、别名等个人配置不干扰教学。

Git 缺失时会给出明确提示；安装后重启 Jupyter 再运行。


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import tempfile

git_exe = shutil.which("git")
if git_exe is None:
    raise RuntimeError("请先安装 Git，加入 PATH，再重启 Jupyter。")

git_env = os.environ.copy()
for key in list(git_env):
    if key.startswith("GIT_"):
        git_env.pop(key)
git_env.update(GIT_CONFIG_NOSYSTEM="1", GIT_CONFIG_GLOBAL=os.devnull,
               GIT_TERMINAL_PROMPT="0", GIT_PAGER="cat",
               GIT_MERGE_AUTOEDIT="no")
lab = Path(tempfile.mkdtemp(prefix="week3_git_lab_")).resolve()

def git(repo, *args, check=True, echo=True):
    repo = Path(repo).resolve()
    if not repo.is_relative_to(lab):
        raise ValueError("命令必须位于本次临时目录内")
    result = subprocess.run(
        [git_exe, "-c", "core.quotepath=false", "-c", "core.autocrlf=false",
         "-c", "commit.gpgsign=false", "-c", "tag.gpgsign=false", *args],
        cwd=repo, env=git_env, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=30)
    if echo:
        print("$ git", " ".join(args))
        if result.stdout.strip():
            print(result.stdout.rstrip())
        if result.stderr.strip():
            print(result.stderr.rstrip())
    if check and result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    return result

def configure(repo):
    git(repo, "config", "--local", "user.name", "Week3 Student", echo=False)
    git(repo, "config", "--local", "user.email", "student@example.invalid", echo=False)

def new_repo(name):
    repo = lab / name
    repo.mkdir()
    git(repo, "init", "-b", "main")
    configure(repo)
    return repo

git(lab, "--version")
print("实验目录：", lab)


$ git --version
git version 2.50.1.windows.1
实验目录： C:\Users\Wyatt\AppData\Local\Temp\week3_git_lab_4eobhvrk


## 4. 暂存与提交：记录的是哪个版本？

先写 version = 1，执行 add，再把工作区改成 version = 2。预测第一次 commit 保存哪一个版本。

<code>git diff</code> 比较工作区与暂存区；<code>git diff --cached</code> 比较暂存区与 HEAD。简短状态的两列分别表示暂存区和工作区的状态。


In [2]:
repo = new_repo("snapshots")
demo = repo / "demo.py"
demo.write_text("version = 1\n", encoding="utf-8")
git(repo, "status", "--short")
git(repo, "add", "demo.py")
demo.write_text("version = 2\n", encoding="utf-8")
git(repo, "status", "--short")
git(repo, "diff")
git(repo, "diff", "--cached")
git(repo, "commit", "-m", "Record version 1")
first_content = git(repo, "show", "HEAD:demo.py").stdout
assert first_content == "version = 1\n"
assert demo.read_text(encoding="utf-8") == "version = 2\n"


$ git init -b main
Initialized empty Git repository in C:/Users/Wyatt/AppData/Local/Temp/week3_git_lab_4eobhvrk/snapshots/.git/
$ git status --short
?? demo.py


$ git add demo.py
$ git status --short
AM demo.py


$ git diff
diff --git a/demo.py b/demo.py
index 6e33cae..952cf60 100644
--- a/demo.py
+++ b/demo.py
@@ -1 +1 @@
-version = 1
+version = 2
$ git diff --cached
diff --git a/demo.py b/demo.py
new file mode 100644
index 0000000..6e33cae
--- /dev/null
+++ b/demo.py
@@ -0,0 +1 @@
+version = 1


$ git commit -m Record version 1
[main (root-commit) 45dd013] Record version 1
 1 file changed, 1 insertion(+)
 create mode 100644 demo.py
$ git show HEAD:demo.py
version = 1


**结果：** add 捕获执行那一刻的内容，后续编辑不会自动进入提交。首次暂存后再修改通常显示 AM；提交后仍有未暂存修改。再次 add、commit 才记录 version = 2。


In [3]:
git(repo, "add", "demo.py")
git(repo, "commit", "-m", "Update demo to version 2")
git(repo, "log", "--oneline")
assert git(repo, "status", "--porcelain", echo=False).stdout == ""
print("两次提交完成，工作区干净。")


$ git add demo.py
$ git commit -m Update demo to version 2
[main cfa69c0] Update demo to version 2
 1 file changed, 1 insertion(+), 1 deletion(-)


$ git log --oneline
cfa69c0 Update demo to version 2
45dd013 Record version 1


两次提交完成，工作区干净。


### .gitignore 与 add 的范围

讲义中的 <code>git add .</code> 会处理当前目录范围内未忽略的新文件、修改和删除，不仅是修改过的文件。忽略规则不会自动取消已跟踪文件。下面观察日志文件显示为 !!，并确认它未被跟踪。


In [4]:
(repo / ".gitignore").write_text("*.log\n", encoding="utf-8")
(repo / "debug.log").write_text("temporary log\n", encoding="utf-8")
git(repo, "status", "--short", "--ignored")
git(repo, "add", ".gitignore")
git(repo, "commit", "-m", "Ignore temporary logs")
assert "debug.log" not in git(repo, "ls-files", echo=False).stdout.splitlines()


$ git status --short --ignored
?? .gitignore
!! debug.log
$ git add .gitignore


$ git commit -m Ignore temporary logs
[main db71ddf] Ignore temporary logs
 1 file changed, 1 insertion(+)
 create mode 100644 .gitignore


## 5. 分支与合并

创建 feature/report 并提交报告，然后切回 main。合并前 main 没有报告文件。使用 --no-ff 明确演示有两个父提交的 merge commit；如果不加该选项且分支没有分叉，Git 可以直接快进。


In [5]:
git(repo, "switch", "-c", "feature/report")
(repo / "report.txt").write_text("Week 3 report\n", encoding="utf-8")
git(repo, "add", "report.txt")
git(repo, "commit", "-m", "Add study report")
git(repo, "switch", "main")
print("合并前 main 有报告吗？", (repo / "report.txt").exists())
assert not (repo / "report.txt").exists()
git(repo, "merge", "--no-ff", "feature/report", "-m", "Merge study report")
git(repo, "log", "--graph", "--oneline", "--all")
parents = git(repo, "show", "-s", "--format=%P", "HEAD", echo=False).stdout.split()
assert len(parents) == 2 and (repo / "report.txt").exists()


$ git switch -c feature/report
Switched to a new branch 'feature/report'
$ git add report.txt


$ git commit -m Add study report
[feature/report 92b295b] Add study report
 1 file changed, 1 insertion(+)
 create mode 100644 report.txt
$ git switch main
Switched to branch 'main'
合并前 main 有报告吗？ False


$ git merge --no-ff feature/report -m Merge study report
Merge made by the 'ort' strategy.
 report.txt | 1 +
 1 file changed, 1 insertion(+)
 create mode 100644 report.txt
$ git log --graph --oneline --all
*   414f456 Merge study report
|\  
| * 92b295b Add study report
|/  
* db71ddf Ignore temporary logs
* cfa69c0 Update demo to version 2
* 45dd013 Record version 1


## 6. 制造并解决合并冲突

两个分支把同一行改成不同值，Git 需要人判断最终结果。下一格故意制造冲突，只在这次 merge 允许非零退出码，并核对冲突文件列表，避免把其他错误误认为成功。


In [6]:
conflict_repo = new_repo("conflicts")
settings = conflict_repo / "settings.txt"
settings.write_text("theme=plain\n", encoding="utf-8")
git(conflict_repo, "add", "settings.txt")
git(conflict_repo, "commit", "-m", "Add default theme")
git(conflict_repo, "switch", "-c", "feature/theme")
settings.write_text("theme=dark\n", encoding="utf-8")
git(conflict_repo, "commit", "-am", "Use dark theme")
git(conflict_repo, "switch", "main")
settings.write_text("theme=light\n", encoding="utf-8")
git(conflict_repo, "commit", "-am", "Use light theme")
result = git(conflict_repo, "merge", "feature/theme", check=False)
unmerged = git(conflict_repo, "diff", "--name-only", "--diff-filter=U", echo=False).stdout.splitlines()
assert result.returncode == 1 and unmerged == ["settings.txt"]
print(settings.read_text(encoding="utf-8"))


$ git init -b main
Initialized empty Git repository in C:/Users/Wyatt/AppData/Local/Temp/week3_git_lab_4eobhvrk/conflicts/.git/


$ git add settings.txt


$ git commit -m Add default theme
[main (root-commit) 06ee369] Add default theme
 1 file changed, 1 insertion(+)
 create mode 100644 settings.txt
$ git switch -c feature/theme
Switched to a new branch 'feature/theme'


$ git commit -am Use dark theme
[feature/theme d1b6d05] Use dark theme
 1 file changed, 1 insertion(+), 1 deletion(-)
$ git switch main
Switched to branch 'main'


$ git commit -am Use light theme
[main 1eadf10] Use light theme
 1 file changed, 1 insertion(+), 1 deletion(-)


$ git merge feature/theme
Auto-merging settings.txt
CONFLICT (content): Merge conflict in settings.txt
Automatic merge failed; fix conflicts and then commit the result.
<<<<<<< HEAD
theme=light
theme=dark
>>>>>>> feature/theme



标记 <<<<<<< 到 ======= 是当前分支的内容，======= 到 >>>>>>> 是另一分支内容。本例决定采用 theme=auto。移除冲突标记、保存文件、add，再 commit；实际协作应根据需求判断，不能机械拼接两份内容。


In [7]:
settings.write_text("theme=auto\n", encoding="utf-8")
git(conflict_repo, "add", "settings.txt")
git(conflict_repo, "commit", "-m", "Resolve theme conflict with auto mode")
git(conflict_repo, "log", "--graph", "--oneline", "--all")
assert git(conflict_repo, "status", "--porcelain", echo=False).stdout == ""
assert settings.read_text(encoding="utf-8") == "theme=auto\n"


$ git add settings.txt


$ git commit -m Resolve theme conflict with auto mode
[main 07bdb23] Resolve theme conflict with auto mode
$ git log --graph --oneline --all
*   07bdb23 Resolve theme conflict with auto mode
|\  
| * d1b6d05 Use dark theme
* | 1eadf10 Use light theme
|/  
* 06ee369 Add default theme


## 7. 撤销：先区分修改阶段

| 情境 | 方法 | 结果 |
|---|---|---|
| 丢弃未暂存的工作区修改 | git restore 文件 | 从暂存区恢复文件，工作区修改会丢失 |
| 取消暂存 | git restore --staged 文件 | 暂存区恢复为 HEAD，工作区保留 |
| 重做最新本地提交 | Desktop Undo；命令行可用 git reset --mixed HEAD~1 | 分支后退，改动留在工作区 |
| 抵消已分享的普通提交 | git revert 哈希 | 新增反向提交，保留原历史 |

reset 的 --soft、--mixed、--hard 对暂存区和工作区的影响不同。revert 撤销指定提交的差异，不保证恢复到任意旧版本的完整状态；撤销较早提交还可能发生冲突。参见 [Desktop 官方说明](https://docs.github.com/en/desktop/managing-commits/reverting-a-commit-in-github-desktop)。


In [8]:
undo_repo = new_repo("undo")
note = undo_repo / "note.txt"
note.write_text("base\n", encoding="utf-8")
git(undo_repo, "add", "note.txt")
git(undo_repo, "commit", "-m", "Create base note")

note.write_text("uncommitted draft\n", encoding="utf-8")
git(undo_repo, "restore", "note.txt")
assert note.read_text(encoding="utf-8") == "base\n"
print("restore：工作区修改已丢弃。")

note.write_text("revised\n", encoding="utf-8")
git(undo_repo, "add", "note.txt")
git(undo_repo, "commit", "-m", "Revise note")
git(undo_repo, "reset", "--mixed", "HEAD~1")
assert note.read_text(encoding="utf-8") == "revised\n"
assert git(undo_repo, "diff", "--cached", echo=False).stdout == ""
assert git(undo_repo, "diff", echo=False).stdout != ""
print("reset --mixed：内容保留在工作区，等待重新暂存。")

git(undo_repo, "add", "note.txt")
git(undo_repo, "commit", "-m", "Resubmit revised note")
old_head = git(undo_repo, "rev-parse", "HEAD", echo=False).stdout.strip()
git(undo_repo, "revert", "--no-edit", "HEAD")
assert note.read_text(encoding="utf-8") == "base\n"
assert git(undo_repo, "rev-parse", "HEAD^", echo=False).stdout.strip() == old_head
git(undo_repo, "log", "--oneline")
print("revert：旧提交仍在，新增一次反向提交。")


$ git init -b main
Initialized empty Git repository in C:/Users/Wyatt/AppData/Local/Temp/week3_git_lab_4eobhvrk/undo/.git/


$ git add note.txt


$ git commit -m Create base note
[main (root-commit) 5a21d88] Create base note
 1 file changed, 1 insertion(+)
 create mode 100644 note.txt
$ git restore note.txt
restore：工作区修改已丢弃。


$ git add note.txt
$ git commit -m Revise note
[main bd4f535] Revise note
 1 file changed, 1 insertion(+), 1 deletion(-)


$ git reset --mixed HEAD~1
Unstaged changes after reset:
M	note.txt


reset --mixed：内容保留在工作区，等待重新暂存。
$ git add note.txt


$ git commit -m Resubmit revised note
[main f52e02e] Resubmit revised note
 1 file changed, 1 insertion(+), 1 deletion(-)


$ git revert --no-edit HEAD
[main c84a7ae] Revert "Resubmit revised note"
 Date: Thu Sep 17 12:59:23 2026 +0800
 1 file changed, 1 insertion(+), 1 deletion(-)


$ git log --oneline
c84a7ae Revert "Resubmit revised note"
f52e02e Resubmit revised note
5a21d88 Create base note
revert：旧提交仍在，新增一次反向提交。


## 8. 本地模拟 clone、push、fetch、pull

bare 仓库保存历史，没有普通工作区。Alice 和 Bob 是两个本地克隆目录，模拟协作者。此实验不包含 GitHub 身份认证、权限和 PR。

push -u origin main 上传 main，并为该分支建立上游关联。origin 是远程名称，main 是本例显式选择的分支名称；讲义中的 master 并非必需。


In [9]:
remote = lab / "remote.git"
git(lab, "init", "--bare", "--initial-branch=main", str(remote))
alice, bob = lab / "alice", lab / "bob"
git(lab, "clone", str(remote), str(alice))
configure(alice)
(alice / "shared.txt").write_text("version 1\n", encoding="utf-8")
git(alice, "add", "shared.txt")
git(alice, "commit", "-m", "Publish first version")
git(alice, "push", "-u", "origin", "main")
git(lab, "clone", str(remote), str(bob))
configure(bob)

(alice / "shared.txt").write_text("version 2 from Alice\n", encoding="utf-8")
git(alice, "commit", "-am", "Publish second version")
git(alice, "push")
git(bob, "fetch", "origin")
assert (bob / "shared.txt").read_text() == "version 1\n"
print("fetch 后工作区仍为 version 1，远程跟踪引用已经更新：")
git(bob, "show", "origin/main:shared.txt")
git(bob, "pull", "--ff-only")
assert (bob / "shared.txt").read_text() == "version 2 from Alice\n"
print("pull 后：", (bob / "shared.txt").read_text().strip())


$ git init --bare --initial-branch=main C:\Users\Wyatt\AppData\Local\Temp\week3_git_lab_4eobhvrk\remote.git
Initialized empty Git repository in C:/Users/Wyatt/AppData/Local/Temp/week3_git_lab_4eobhvrk/remote.git/


$ git clone C:\Users\Wyatt\AppData\Local\Temp\week3_git_lab_4eobhvrk\remote.git C:\Users\Wyatt\AppData\Local\Temp\week3_git_lab_4eobhvrk\alice
Cloning into 'C:\Users\Wyatt\AppData\Local\Temp\week3_git_lab_4eobhvrk\alice'...
done.


$ git add shared.txt


$ git commit -m Publish first version
[main (root-commit) 3e2a96d] Publish first version
 1 file changed, 1 insertion(+)
 create mode 100644 shared.txt


$ git push -u origin main
branch 'main' set up to track 'origin/main'.
To C:\Users\Wyatt\AppData\Local\Temp\week3_git_lab_4eobhvrk\remote.git
 * [new branch]      main -> main
$ git clone C:\Users\Wyatt\AppData\Local\Temp\week3_git_lab_4eobhvrk\remote.git C:\Users\Wyatt\AppData\Local\Temp\week3_git_lab_4eobhvrk\bob
Cloning into 'C:\Users\Wyatt\AppData\Local\Temp\week3_git_lab_4eobhvrk\bob'...
done.


$ git commit -am Publish second version
[main 95d6d57] Publish second version
 1 file changed, 1 insertion(+), 1 deletion(-)


$ git push
To C:\Users\Wyatt\AppData\Local\Temp\week3_git_lab_4eobhvrk\remote.git
   3e2a96d..95d6d57  main -> main
$ git fetch origin
From C:\Users\Wyatt\AppData\Local\Temp\week3_git_lab_4eobhvrk\remote
   3e2a96d..95d6d57  main       -> origin/main
fetch 后工作区仍为 version 1，远程跟踪引用已经更新：


$ git show origin/main:shared.txt
version 2 from Alice
$ git pull --ff-only
Updating 3e2a96d..95d6d57
Fast-forward
 shared.txt | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
pull 后： version 2 from Alice


**观察：** fetch 更新远程跟踪引用，不直接改工作区；pull 先 fetch，再按配置或选项整合。本例 --ff-only 要求快进，否则停止。

真实 GitHub 的命令对应关系如下，仅供阅读；先在 GitHub 建立目标仓库，将占位符换成真实值。若 origin 已存在，应先检查它，而非重复 add。

~~~bash
git remote -v
git remote add origin https://github.com/<username>/<repository>.git
git push -u origin main
git push origin feature/report
git pull --ff-only origin main
~~~


## 9. merge 与 rebase 的区别

~~~text
分叉： A -- B -- C  (main)
             \-- D -- E  (feature)

merge：A -- B -- C ------ M  (main)
             \-- D -- E --/   M 有两个父提交

rebase：A -- B -- C -- D' -- E'  (feature)
~~~

普通 rebase 将功能分支独有的提交重新应用到新基底，通常生成新哈希的 D'、E'，不是创建两父节点的 merge commit。原讲义对此的表述需要修正。参见 [Git rebase 官方文档](https://git-scm.com/docs/git-rebase)。

下面两条分支修改不同文件，观察提交重放，不混入冲突处理。


In [10]:
rebase_repo = new_repo("rebase")
(rebase_repo / "base.txt").write_text("base\n", encoding="utf-8")
git(rebase_repo, "add", ".")
git(rebase_repo, "commit", "-m", "Base")
git(rebase_repo, "switch", "-c", "feature")
(rebase_repo / "feature.txt").write_text("feature work\n", encoding="utf-8")
git(rebase_repo, "add", ".")
git(rebase_repo, "commit", "-m", "Feature work")
before = git(rebase_repo, "rev-parse", "HEAD", echo=False).stdout.strip()
git(rebase_repo, "switch", "main")
(rebase_repo / "main.txt").write_text("main work\n", encoding="utf-8")
git(rebase_repo, "add", ".")
git(rebase_repo, "commit", "-m", "Main work")
git(rebase_repo, "switch", "feature")
git(rebase_repo, "rebase", "main")
after = git(rebase_repo, "rev-parse", "HEAD", echo=False).stdout.strip()
parent = git(rebase_repo, "show", "-s", "--format=%P", "HEAD", echo=False).stdout.split()
main_head = git(rebase_repo, "rev-parse", "main", echo=False).stdout.strip()
assert before != after and parent == [main_head]
print("哈希变化：", before[:8], "->", after[:8])
git(rebase_repo, "log", "--graph", "--oneline", "--all")


$ git init -b main
Initialized empty Git repository in C:/Users/Wyatt/AppData/Local/Temp/week3_git_lab_4eobhvrk/rebase/.git/
$ git add .


$ git commit -m Base
[main (root-commit) 0f230cf] Base
 1 file changed, 1 insertion(+)
 create mode 100644 base.txt
$ git switch -c feature
Switched to a new branch 'feature'
$ git add .


$ git commit -m Feature work
[feature b01332a] Feature work
 1 file changed, 1 insertion(+)
 create mode 100644 feature.txt
$ git switch main
Switched to branch 'main'
$ git add .


$ git commit -m Main work
[main f5f7af8] Main work
 1 file changed, 1 insertion(+)
 create mode 100644 main.txt
$ git switch feature
Switched to branch 'feature'


$ git rebase main
Rebasing (1/1)
Successfully rebased and updated refs/heads/feature.
哈希变化： b01332ad -> 92602b73
$ git log --graph --oneline --all
* 92602b7 Feature work
* f5f7af8 Main work
* 0f230cf Base


CompletedProcess(args=['D:\\Git\\cmd\\git.EXE', '-c', 'core.quotepath=false', '-c', 'core.autocrlf=false', '-c', 'commit.gpgsign=false', '-c', 'tag.gpgsign=false', 'log', '--graph', '--oneline', '--all'], returncode=0, stdout='* 92602b7 Feature work\n* f5f7af8 Main work\n* 0f230cf Base\n', stderr='')

### 补充：原讲义图中的 squash merge

原文第四张图还比较了 **Squash Commit**：把功能分支的最终差异压成一个新提交。它不会把功能分支的每个提交都接入 main 的祖先链，也不是普通的两父节点 merge commit。

| 方式 | 历史效果 |
|---|---|
| fast-forward | 直接移动分支引用，不创建合并提交 |
| merge --no-ff | 创建具有两个父提交的合并提交 |
| rebase | 在新基底上重放独有提交，通常产生新哈希 |
| merge --squash 后 commit | 将合并结果暂存，再记录为一个单父提交 |

下一格创建两个功能提交，再把它们压成 main 上的一个提交。观察 squash 操作只准备内容，仍需显式 commit。

In [11]:
squash_repo = new_repo("squash")
(squash_repo / "base.txt").write_text("base", encoding="utf-8")
git(squash_repo, "add", ".")
git(squash_repo, "commit", "-m", "Base")
git(squash_repo, "switch", "-c", "feature")
for number in (1, 2):
    (squash_repo / f"part{number}.txt").write_text(f"part {number}", encoding="utf-8")
    git(squash_repo, "add", ".")
    git(squash_repo, "commit", "-m", f"Add part {number}")
git(squash_repo, "switch", "main")
git(squash_repo, "merge", "--squash", "feature")
assert git(squash_repo, "rev-list", "--count", "HEAD", echo=False).stdout.strip() == "1"
git(squash_repo, "commit", "-m", "Add complete feature as one commit")
assert git(squash_repo, "rev-list", "--count", "HEAD", echo=False).stdout.strip() == "2"
assert len(git(squash_repo, "show", "-s", "--format=%P", "HEAD", echo=False).stdout.split()) == 1
git(squash_repo, "log", "--graph", "--oneline", "--all")
print("main：初始提交 + 一个 squash 提交；功能分支仍保留自己的两个提交。")

$ git init -b main
Initialized empty Git repository in C:/Users/Wyatt/AppData/Local/Temp/week3_git_lab_4eobhvrk/squash/.git/


$ git add .


$ git commit -m Base
[main (root-commit) 90d7ef0] Base
 1 file changed, 1 insertion(+)
 create mode 100644 base.txt
$ git switch -c feature
Switched to a new branch 'feature'


$ git add .


$ git commit -m Add part 1
[feature f3a036c] Add part 1
 1 file changed, 1 insertion(+)
 create mode 100644 part1.txt
$ git add .


$ git commit -m Add part 2
[feature a390b8e] Add part 2
 1 file changed, 1 insertion(+)
 create mode 100644 part2.txt


$ git switch main
Switched to branch 'main'
$ git merge --squash feature
Updating 90d7ef0..a390b8e
Fast-forward
Squash commit -- not updating HEAD
 part1.txt | 1 +
 part2.txt | 1 +
 2 files changed, 2 insertions(+)
 create mode 100644 part1.txt
 create mode 100644 part2.txt


$ git commit -m Add complete feature as one commit
[main cf09045] Add complete feature as one commit
 2 files changed, 2 insertions(+)
 create mode 100644 part1.txt
 create mode 100644 part2.txt
$ git log --graph --oneline --all
* cf09045 Add complete feature as one commit
| * a390b8e Add part 2
| * f3a036c Add part 1
|/  
* 90d7ef0 Base
main：初始提交 + 一个 squash 提交；功能分支仍保留自己的两个提交。


## 10. 动手练习与参考实现

先自己尝试：新建练习仓库，提交含 70,80 的 scores.txt；创建 feature/score，增加 90 并提交；合并到 main，检查最终内容和工作区。


In [12]:
practice = new_repo("practice")
scores = practice / "scores.txt"
scores.write_text("70,80\n", encoding="utf-8")
git(practice, "add", "scores.txt")
git(practice, "commit", "-m", "Add initial scores")
git(practice, "switch", "-c", "feature/score")
scores.write_text("70,80,90\n", encoding="utf-8")
git(practice, "commit", "-am", "Add third score")
git(practice, "switch", "main")
git(practice, "merge", "--ff-only", "feature/score")
assert scores.read_text(encoding="utf-8") == "70,80,90\n"
assert git(practice, "status", "--porcelain", echo=False).stdout == ""
print("✓ 练习通过：成绩更新已进入 main。")


$ git init -b main
Initialized empty Git repository in C:/Users/Wyatt/AppData/Local/Temp/week3_git_lab_4eobhvrk/practice/.git/


$ git add scores.txt


$ git commit -m Add initial scores
[main (root-commit) fe387ae] Add initial scores
 1 file changed, 1 insertion(+)
 create mode 100644 scores.txt
$ git switch -c feature/score
Switched to a new branch 'feature/score'


$ git commit -am Add third score
[feature/score 3ad0cd5] Add third score
 1 file changed, 1 insertion(+), 1 deletion(-)


$ git switch main
Switched to branch 'main'
$ git merge --ff-only feature/score
Updating fe387ae..3ad0cd5
Fast-forward
 scores.txt | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
✓ 练习通过：成绩更新已进入 main。


## 11. 复习自测

1. 保存文件、commit、push 各完成什么？
2. add 后又编辑文件，第一次提交保存哪个版本？
3. 合并到 main 应先切到哪条分支？
4. revert 为什么适合撤销已分享提交？
5. fetch 和 pull 对工作区有何区别？
6. 普通 rebase 是否会创建 merge commit？

<details><summary>参考答案</summary>

1. 修改工作区、保存本地快照、上传提交。
2. add 时暂存的版本。
3. main。
4. 它增加反向提交，保留已有历史。
5. fetch 不直接整合工作区，pull 会尝试整合。
6. 通常不会；它重放提交并改变其基底。

</details>

请 Restart Kernel and Run All。临时仓库保留供查看历史；每次完整运行使用新目录，章节内的状态修改命令请按顺序执行。


In [13]:
assert first_content == "version = 1\n"
assert settings.read_text(encoding="utf-8") == "theme=auto\n"
assert note.read_text(encoding="utf-8") == "base\n"
assert before != after
assert (bob / "shared.txt").read_text() == "version 2 from Alice\n"
print("✓ Lecture 03：提交、分支、冲突、撤销、远程同步、变基全部通过。")


✓ Lecture 03：提交、分支、冲突、撤销、远程同步、变基全部通过。
